# 冻结 Stage1 与同口径强参考的 126 日窗口交集

只读原生 Qlib 0bp Tiered 账户，不选择或运行切换策略。

In [1]:
import hashlib,json,os,pathlib,math
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=pathlib.Path.cwd();card=root/'research/stage1_strong_reference_window_overlap_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
a=root/src['matched_reference_card'];b=root/src['stage1_component_card'];asum=json.loads((a/'summary.json').read_text());bsum=json.loads((b/'summary.json').read_text())
assert sha(a/'summary.json')==src['matched_reference_summary_sha256'] and sha(b/'summary.json')==src['stage1_component_summary_sha256']
assert asum['recorder_id']==src['reference_recorder_id'] and asum['experiment_id']==src['reference_experiment_id'] and asum['nas_evidence']['sha256']==src['reference_NAS_sha256']
assert bsum['recorder_id']==src['component_recorder_id'] and bsum['experiment_id']==src['component_experiment_id'] and bsum['nas_evidence']['sha256']==src['component_NAS_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/stage1_domain_heads_20261018'),region='us',kernels=1)
with R.uri_context(str(a/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['reference_experiment_id'],recorder_id=src['reference_recorder_id']);assert rec.status=='FINISHED'
 stage=rec.load_object('analysis/rolling/A_active_research_replay_0');strong=rec.load_object('analysis/rolling/strong9097_0')
with R.uri_context(str(b/'qlib_recorder')):
 rec=R.get_recorder(experiment_id=src['component_experiment_id'],recorder_id=src['component_recorder_id']);assert rec.status=='FINISHED'
 comp=rec.load_object('analysis/rolling126')
assert len(stage)==len(strong)==len(comp)==819
for field in ('first_trade','last_exit'):
 a_date=pd.to_datetime(stage[field]).dt.normalize();s_date=pd.to_datetime(strong[field]).dt.normalize();c_date=pd.to_datetime(comp[field]).dt.normalize()
 assert a_date.equals(s_date) and a_date.equals(c_date)
assert np.max(np.abs(stage.relative_log.to_numpy(float)-comp.relative_log_excess.to_numpy(float)))<1e-9
wa=stage.relative_log.to_numpy(float)>0;ws=strong.relative_log.to_numpy(float)>0
cash=comp.cash_opportunity_log.to_numpy(float);selection=comp.SOXL_selection_log.to_numpy(float)
masks={'both_positive':wa&ws,'Stage1_only_positive':wa&~ws,'strong_only_positive':~wa&ws,'neither_positive':~wa&~ws}
groups={}
for name,mask in masks.items():
 groups[name]={'windows':int(mask.sum()),'Stage1_cash_negative':int((cash[mask]<0).sum()),'Stage1_selection_negative':int((selection[mask]<0).sum()),'Stage1_mean_cash_log':float(cash[mask].mean()) if mask.any() else None,'Stage1_mean_selection_log':float(selection[mask].mean()) if mask.any() else None}
need=math.ceil(.70*len(stage));union=int((wa|ws).sum())
result={'decision':'descriptive_same_basis_Stage1_vs_strong_reference_window_overlap_no_policy','plan_sha256':sha(card/'plan.json'),'windows':len(stage),'positive_required_70pct':need,'Stage1_positive':int(wa.sum()),'strong9097_positive':int(ws.sum()),'groups':groups,'hindsight_whole_window_union_positive':union,'hindsight_union_covers_70pct':union>=need,'minimum_additional_positive_beyond_union_if_below_requirement':max(0,need-union),'original_Stage1_component_relative_log_max_error':float(np.max(np.abs(stage.relative_log.to_numpy(float)-comp.relative_log_excess.to_numpy(float)))),'windows_overlap':True,'hindsight_union_is_tradable':False,'new_market_requests':0,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n');print(json.dumps(result,ensure_ascii=False,indent=2))


{
  "decision": "descriptive_same_basis_Stage1_vs_strong_reference_window_overlap_no_policy",
  "plan_sha256": "c2be53688879eb7f981719ce506bad523b169422ded04d3abd9429aafe0dedd7",
  "windows": 819,
  "positive_required_70pct": 574,
  "Stage1_positive": 255,
  "strong9097_positive": 481,
  "groups": {
    "both_positive": {
      "windows": 251,
      "Stage1_cash_negative": 92,
      "Stage1_selection_negative": 69,
      "Stage1_mean_cash_log": 0.00031639371466617573,
      "Stage1_mean_selection_log": 0.0691435875226492
    },
    "Stage1_only_positive": {
      "windows": 4,
      "Stage1_cash_negative": 2,
      "Stage1_selection_negative": 1,
      "Stage1_mean_cash_log": 0.007305486553662957,
      "Stage1_mean_selection_log": 0.004373046159043139
    },
    "strong_only_positive": {
      "windows": 230,
      "Stage1_cash_negative": 205,
      "Stage1_selection_negative": 56,
      "Stage1_mean_cash_log": -0.08965499905706595,
      "Stage1_mean_selection_log": 0.034860464812777

## 中文结论

原 Stage1 255/819、同口径强参考 481/819 个正超额窗口，共同 251、Stage1 独有4、强参考独有230、两者均不赢334。事后并集485，仍差574门槛89个；均不赢窗口中 Stage1 现金成本331个为负。此并集不是可交易策略或原生混合账户。零拟合、零新账户、不晋级。